# Coronary Stenosis — Data Exploration
**Input:** `roi_features_subset_b.csv` produced by the 1498-image feature extraction pipeline.

**Key facts about the CSV structure:**
- `roi_name` = `{patient_id}_{serie_id}_{frame}_{roi_idx}` where `roi_idx` ∈ [1, 100] are skeleton-sampled ROIs and `roi_idx ≥ 101` are forced GT-centred boxes (always stenosis=1 — must be **excluded from the test split**)
- 2191 feature columns: 30 raw (global + 4 tiles × 6 stats) + 2160 Gabor (72 filters × 30) + 1 `width_ratio`
- Target: `label` — 1 = stenosis, 0 = healthy

## 0 · Imports & config

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import f_classif
from matplotlib.patches import Patch

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120})

# ── Update this path ──────────────────────────────────────────────────────────
CSV_PATH  = r"C:\Users\mluser\IPA_ML\notebooks\notebooks\csv_files\subset_1498images\roi_features_subset_b.csv"

# ── Feature structure (must match feature_extraction constants) ───────────────
STAT_NAMES  = ['mean', 'var', 'entropy', 'energy', 'kurtosis', 'skewness']
TILE_LABELS = ['tl', 'tr', 'bl', 'br']
N_SIZES        = 6
N_ORIENTATIONS = 12
N_FILTERS      = N_SIZES * N_ORIENTATIONS  # 72

META_COLS   = ['roi_name']
TARGET_COL  = 'label'

# ROI index threshold: roi_idx > FORCED_ROI_THRESHOLD are forced GT boxes
FORCED_ROI_THRESHOLD = 100

## 1 · Load & parse roi_name

In [2]:
df_all = pd.read_csv(CSV_PATH)

# Parse roi_name → patient_id, serie_id, frame_id, roi_idx
# Format: {patient_id}_{serie_id}_{frame_stem}_{roi_idx}
# roi_idx is the LAST token (always an integer)
def parse_roi_name(name):
    # Format: {patient}_{serie}_14_{patient}_{serie}_{frame}_{roi_idx}
    # tokens:    [0]      [1]  [2]    [3]      [4]     [5]      [-1]
    parts      = str(name).split('_')
    patient_id = parts[0]    # '002'
    serie_id   = parts[1]    # '5'
    frame_id   = parts[5]    # '0034'
    roi_idx    = int(parts[-1])  # 1
    return patient_id, serie_id, frame_id, roi_idx

parsed = df_all['roi_name'].apply(parse_roi_name)
df_all[['patient_id','serie_id','frame_id','roi_idx']] = pd.DataFrame(
    parsed.tolist(), index=df_all.index
)

# Flag forced GT boxes (roi_idx > 100)
df_all['is_forced_gt'] = df_all['roi_idx'] > FORCED_ROI_THRESHOLD

feature_cols = [c for c in df_all.columns
                if c not in META_COLS + [TARGET_COL,
                   'patient_id','serie_id','frame_id','roi_idx','is_forced_gt']]

print(f'Total ROIs        : {len(df_all):,}')
print(f'  skeleton-sampled: {(~df_all.is_forced_gt).sum():,}  (roi_idx 1-100)')
print(f'  forced GT boxes : {df_all.is_forced_gt.sum():,}   (roi_idx > 100, always stenosis=1)')
print(f'Feature columns   : {len(feature_cols)}')
print(f'Unique patients   : {df_all.patient_id.nunique()}')
print(f'Unique series     : {df_all.serie_id.nunique()}')
df_all.head(3)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\mluser\\IPA_ML\\notebooks\\notebooks\\csv_files\\subset_1498images\\roi_features_subset_b.csv'

## 2 · df.info() and describe()

In [ ]:
# 1 serie aleatoria por paciente
selected_series = (
    df_all.groupby('patient_id')['serie_id']
    .apply(lambda s: s.drop_duplicates()
                      .sample(n=1, random_state=RANDOM_STATE)
                      .iloc[0])
    .reset_index()
    .rename(columns={'serie_id': 'selected_serie'})
)

df = df_all.merge(
    selected_series,
    left_on  = ['patient_id', 'serie_id'],
    right_on = ['patient_id', 'selected_serie']
).drop(columns='selected_serie').reset_index(drop=True)

sampled = df[~df.is_forced_gt].copy()

print(f'Antes : {len(df_all):,} ROIs — {df_all.groupby(["patient_id","serie_id"]).ngroups} series')
print(f'Después: {len(df):,} ROIs — {df.patient_id.nunique()} pacientes × 1 serie')

In [ ]:
df[feature_cols + [TARGET_COL]].info(verbose=False, show_counts=True)

In [ ]:
df[feature_cols].describe().T \
    .sort_values('std', ascending=False) \
    .head(20)

## 3 · Missing values

In [ ]:
nan_per_col = df[feature_cols].isna().sum()
nan_pct     = nan_per_col / len(df) * 100
nan_df      = pd.DataFrame({'n_missing': nan_per_col, 'pct_missing': nan_pct})
nan_df      = nan_df[nan_df.n_missing > 0].sort_values('pct_missing', ascending=False)

print(f'Features with NaNs: {len(nan_df)} / {len(feature_cols)}')

if len(nan_df):
    display(nan_df.head(20))
    # NaN rows by class & forced-GT flag
    nan_rows = df[df[feature_cols].isna().any(axis=1)]
    print('\nNaN rows by label:')
    print(nan_rows[TARGET_COL].value_counts().rename({0:'Healthy',1:'Stenosis'}))
    print('\nNaN rows are forced GT?')
    print(nan_rows['is_forced_gt'].value_counts())

    # Most problematic features
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.bar(range(len(nan_df)), nan_df.pct_missing, color='#E85D24', alpha=0.8)
    ax.set_xlabel('Feature (sorted by % missing)')
    ax.set_ylabel('% missing')
    ax.set_title('Missing values per feature')
    plt.tight_layout(); plt.show()
else:
    print('✓ No missing values.')

## 4 · Forced GT boxes — the test-split contamination issue

ROIs with `roi_idx > 100` are forced GT-centred boxes injected during extraction so that every stenosis always has at least one positive ROI. They are **always label=1 by construction** and must be **removed from the test split** (they are not blind predictions — the model would see data it was designed to find).

Below we verify their properties and create the clean training-eligible DataFrame.

In [ ]:
forced  = df[df.is_forced_gt]
sampled = df[~df.is_forced_gt]

print('=== Forced GT boxes (roi_idx > 100) ===')
print(f'  Count                : {len(forced):,}')
print(f'  label=1 (stenosis)   : {forced[TARGET_COL].sum():,}  ({forced[TARGET_COL].mean()*100:.1f}%)')
print(f'  label=0 (healthy)    : {(forced[TARGET_COL]==0).sum():,}  — should be 0')
print()
print('=== Skeleton-sampled ROIs (roi_idx 1-100) ===')
print(f'  Count                : {len(sampled):,}')
print(f'  label=1 (stenosis)   : {sampled[TARGET_COL].sum():,}  ({sampled[TARGET_COL].mean()*100:.2f}%)')
print(f'  label=0 (healthy)    : {(sampled[TARGET_COL]==0).sum():,}')
print()

# How many forced GT per image on average?
forced_per_image = forced.groupby(['patient_id','serie_id','frame_id']).size()
print(f'Forced GT boxes per image: mean={forced_per_image.mean():.2f}, max={forced_per_image.max()}')

# Recommendation
print()
print('ACTION: For the test split, use ONLY sampled ROIs (roi_idx <= 100).')
print('        Forced GT boxes may be kept in the TRAINING set to boost positive class.')

## 5 · Class distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

COLORS = {'Healthy': '#5DCAA5', 'Stenosis': '#E85D24'}

for ax, subset, title in zip(
    axes,
    [df, sampled, forced],
    ['All ROIs', 'Skeleton-sampled only (model-eligible)', 'Forced GT boxes']
):
    vc = subset[TARGET_COL].value_counts().sort_index()
    labels_map = {0:'Healthy', 1:'Stenosis'}
    names  = [labels_map[k] for k in vc.index]
    counts = vc.values
    colors = [COLORS[n] for n in names]
    bars = ax.bar(names, counts, color=colors, width=0.5)
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + len(subset)*0.003,
                f'{count:,}', ha='center', va='bottom', fontsize=10)
    ratio = vc.get(0, 0) / max(vc.get(1, 1), 1)
    ax.set_title(f'{title}\nimbalance {ratio:.1f}:1', fontsize=10)
    ax.set_ylabel('ROI count')

plt.suptitle('Class distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Per-patient stenosis ROI count (skeleton-sampled only)
per_patient = sampled.groupby('patient_id')[TARGET_COL].agg(
    n_stenosis='sum', n_total='count'
)
per_patient['n_healthy']    = per_patient.n_total - per_patient.n_stenosis
per_patient['stenosis_pct'] = per_patient.n_stenosis / per_patient.n_total * 100

zero_stenosis_patients = (per_patient.n_stenosis == 0).sum()
print(f'Patients with ZERO stenosis ROIs (skeleton-sampled): {zero_stenosis_patients}')
print('→ These are the patients relevant to Bria\'s sensitivity question.')
print()
display(per_patient[per_patient.n_stenosis == 0][['n_total','n_healthy','n_stenosis']])

fig, ax = plt.subplots(figsize=(16, 4))
per_patient_sorted = per_patient.sort_values('stenosis_pct', ascending=False)
x = range(len(per_patient_sorted))
ax.bar(x, per_patient_sorted.n_healthy,  color='#5DCAA5', label='Healthy')
ax.bar(x, per_patient_sorted.n_stenosis, bottom=per_patient_sorted.n_healthy,
       color='#E85D24', label='Stenosis')
ax.set_xticks(list(x))
ax.set_xticklabels(per_patient_sorted.index, rotation=45, ha='right', fontsize=7)
ax.set_xlabel('Patient (sorted by stenosis %)')
ax.set_ylabel('ROI count')
ax.set_title('ROI class distribution per patient (skeleton-sampled ROIs only)')
ax.legend()
plt.tight_layout()
plt.show()

## 6 · Feature group breakdown

Column naming convention: `raw_global_{stat}`, `raw_{tile}_{stat}`, `gabor_f{i}_global_{stat}`, `gabor_f{i}_{tile}_{stat}`, `width_ratio`

In [ ]:
def classify_col(col):
    if col == 'width_ratio':
        return 'raw', 'width_ratio', 'width_ratio'
    parts  = col.split('_')
    source = parts[0]           # 'raw' or 'gabor'
    stat   = parts[-1]          # last token is always the stat name
    # region is everything between source and stat
    middle = parts[1:-1]        # e.g. ['global'] or ['f12', 'tl'] or ['f12', 'global']
    if 'global' in middle:
        region = 'global'
    else:
        region = next((t for t in middle if t in TILE_LABELS), 'other')
    return source, region, stat

meta_df = pd.DataFrame(
    [classify_col(c) for c in feature_cols],
    columns=['source','region','stat'],
    index=feature_cols
)

print('Feature count by source:')
print(meta_df.source.value_counts().to_string())
print()
print('Feature count by region:')
print(meta_df.region.value_counts().to_string())
print()
print('Feature count by stat:')
print(meta_df.stat.value_counts().to_string())

## 7 · Correlation with target label

In [ ]:
# Work on skeleton-sampled ROIs only for unbiased analysis
corr = sampled[feature_cols].corrwith(sampled[TARGET_COL]).sort_values(key=abs, ascending=False)

print('Top 25 features most correlated with stenosis label (skeleton-sampled ROIs):')
print(corr.head(25).to_string())

In [ ]:
# Bar chart — top 30 by |correlation|
top30 = corr.abs().nlargest(30)
bar_colors = ['#E85D24' if corr[f] > 0 else '#3B8BD4' for f in top30.index]

fig, ax = plt.subplots(figsize=(14, 7))
ax.barh(range(len(top30)), corr[top30.index].values, color=bar_colors)
ax.set_yticks(range(len(top30)))
ax.set_yticklabels(top30.index, fontsize=8)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson correlation with stenosis label')
ax.set_title('Top 30 features by |correlation| with target (skeleton-sampled ROIs)')
ax.invert_yaxis()
ax.legend(handles=[
    Patch(color='#E85D24', label='Positive (higher in stenosis)'),
    Patch(color='#3B8BD4', label='Negative (lower in stenosis)')
])
plt.tight_layout()
plt.show()

In [ ]:
# Correlation broken down by stat type — which stat family is most discriminative?
stat_corr = {}
for stat in STAT_NAMES + ['width_ratio']:
    stat_cols = meta_df[meta_df.stat == stat].index.tolist()
    if stat_cols:
        stat_corr[stat] = corr[stat_cols].abs().mean()

stat_corr_s = pd.Series(stat_corr).sort_values(ascending=False)
print('Mean |correlation| with label per stat family:')
print(stat_corr_s.to_string())

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(stat_corr_s.index, stat_corr_s.values, color='#3B8BD4', alpha=0.8)
ax.set_ylabel('Mean |Pearson r| with label')
ax.set_title('Average discriminative power per stat family')
plt.tight_layout()
plt.show()

## 8 · ANOVA F-test — univariate feature ranking

In [ ]:
X_sampled = sampled[feature_cols].fillna(sampled[feature_cols].median())
y_sampled = sampled[TARGET_COL]

f_scores, p_values = f_classif(X_sampled, y_sampled)

anova_df = pd.DataFrame({
    'feature': feature_cols,
    'F_score': f_scores,
    'p_value': p_values
}).sort_values('F_score', ascending=False).reset_index(drop=True)

sig = (anova_df.p_value < 0.05).sum()
print(f'Features significant at p<0.05: {sig} / {len(feature_cols)}')
print(f'\nTop 20 by ANOVA F-score:')
display(anova_df.head(20))

In [ ]:
# F-score mean per stat family
anova_df['stat'] = anova_df.feature.apply(lambda c: classify_col(c)[2])
anova_df['source'] = anova_df.feature.apply(lambda c: classify_col(c)[0])
anova_df['region'] = anova_df.feature.apply(lambda c: classify_col(c)[1])

print('Mean F-score per stat family:')
print(anova_df.groupby('stat')['F_score'].mean().sort_values(ascending=False).to_string())
print()
print('Mean F-score raw vs gabor:')
print(anova_df.groupby('source')['F_score'].mean().sort_values(ascending=False).to_string())
print()
print('Mean F-score global vs tiles:')
print(anova_df.groupby('region')['F_score'].mean().sort_values(ascending=False).to_string())

In [ ]:
# Boxplots for top 12 features
top12 = anova_df.head(12).feature.tolist()
plot_df = sampled[top12 + [TARGET_COL]].melt(id_vars=TARGET_COL, var_name='feature', value_name='value')
plot_df['class'] = plot_df[TARGET_COL].map({0:'Healthy', 1:'Stenosis'})

g = sns.FacetGrid(plot_df, col='feature', col_wrap=4, height=3, aspect=1.1, sharey=False)
g.map_dataframe(sns.boxplot, x='class', y='value',
                palette={'Healthy':'#5DCAA5','Stenosis':'#E85D24'},
                linewidth=0.7, fliersize=2)
g.set_titles(col_template='{col_name}', size=7)
g.set_axis_labels('', 'value')
plt.suptitle('Top 12 features by ANOVA F-score (skeleton-sampled ROIs)', y=1.02,
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### Feature Selection with Random Forest

In [ ]:
## 15 · Train/Test split + RF Feature Selection

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.preprocessing import StandardScaler

# ── 1. Split a nivel PACIENTE (evita leakage) ────────────────────────────────
patients = sampled['patient_id'].unique()
patients_train, patients_test = train_test_split(
    patients, test_size=0.2, random_state=RANDOM_STATE
)

# Train: sampled + forced GT (señal positiva extra)
train_df = df[df['patient_id'].isin(patients_train)].copy()
# Test: SOLO sampled, sin forced GT
test_df  = sampled[sampled['patient_id'].isin(patients_test)].copy()

print(f'Train ROIs : {len(train_df):,}  '
      f'(stenosis={train_df[TARGET_COL].sum():,})')
print(f'Test ROIs  : {len(test_df):,}   '
      f'(stenosis={test_df[TARGET_COL].sum():,})')
print(f'Train patients: {len(patients_train)} | Test patients: {len(patients_test)}')

# ── 2. Preparar X/y ──────────────────────────────────────────────────────────
X_train = train_df[feature_cols].fillna(train_df[feature_cols].median())
y_train = train_df[TARGET_COL]

X_test  = test_df[feature_cols].fillna(X_train.median())  # mediana del train
y_test  = test_df[TARGET_COL]

# ── 3. Escalar ────────────────────────────────────────────────────────────────
scaler   = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# ── 4. RF Feature Importance ──────────────────────────────────────────────────
rf_selector = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',   # importante por el desbalanceo
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_selector.fit(X_train_scaled, y_train)

importances = pd.Series(rf_selector.feature_importances_, index=feature_cols)
importances_sorted = importances.sort_values(ascending=False)

# Top 50 features
TOP_K = 50
top_features = importances_sorted.head(TOP_K).index.tolist()

print(f'\nTop {TOP_K} features seleccionadas por RF:')
print(importances_sorted.head(TOP_K).to_string())

# ── 5. Visualizar importancias ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 8))
ax.barh(range(TOP_K), importances_sorted.head(TOP_K).values[::-1], color='#3B8BD4')
ax.set_yticks(range(TOP_K))
ax.set_yticklabels(importances_sorted.head(TOP_K).index[::-1], fontsize=7)
ax.set_xlabel('RF Feature Importance')
ax.set_title(f'Top {TOP_K} features por RF Importance')
plt.tight_layout()
plt.show()

# ── 6. Aplicar selección ──────────────────────────────────────────────────────
X_train_sel = X_train_scaled[:, [feature_cols.index(f) for f in top_features]]
X_test_sel  = X_test_scaled[:,  [feature_cols.index(f) for f in top_features]]

print(f'\nShape train después de selección: {X_train_sel.shape}')
print(f'Shape test  después de selección: {X_test_sel.shape}')

## 9 · width_ratio — the stenosis-specific feature

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, subset, title in zip(axes,
                              [sampled, df],
                              ['Skeleton-sampled ROIs', 'All ROIs (incl. forced GT)']):
    for label, color in [(0,'#5DCAA5'),(1,'#E85D24')]:
        vals = subset[subset[TARGET_COL]==label]['width_ratio'].dropna()
        ax.hist(vals, bins=40, alpha=0.6, color=color,
                label='Healthy' if label==0 else 'Stenosis', density=True)
    ax.set_xlabel('width_ratio (max_w / min_w)')
    ax.set_ylabel('density')
    ax.set_title(title)
    ax.legend()

plt.suptitle('width_ratio distribution by class', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# Stats
for label, name in [(0,'Healthy'),(1,'Stenosis')]:
    vals = sampled[sampled[TARGET_COL]==label]['width_ratio'].dropna()
    print(f'{name:10s}: mean={vals.mean():.3f}  median={vals.median():.3f}  std={vals.std():.3f}')

In [ ]:
# Calcular límite razonable (percentil 99 de todos los valores)
p99 = sampled['width_ratio'].quantile(0.99)
print(f'Percentil 99 width_ratio: {p99:.2f}')
print(f'Máximo absoluto         : {sampled["width_ratio"].max():.2f}')
print(f'Valores > p99           : {(sampled["width_ratio"] > p99).sum()}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, subset, title in zip(axes,
                              [sampled, df],
                              ['Skeleton-sampled ROIs', 'All ROIs (incl. forced GT)']):
    for label, color in [(0,'#5DCAA5'),(1,'#E85D24')]:
        vals = subset[subset[TARGET_COL]==label]['width_ratio'].dropna()
        ax.hist(vals, bins=40, alpha=0.6, color=color,
                label='Healthy' if label==0 else 'Stenosis', density=True)
    ax.set_xlabel('width_ratio (max_w / min_w)')
    ax.set_ylabel('density')
    ax.set_title(title)
    ax.set_xlim(1, p99)   # ← limitar eje X
    ax.legend()

plt.suptitle('width_ratio distribution by class', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 10 · Feature-feature correlation heatmap

In [ ]:
# Use top 40 features by F-score to keep the heatmap readable
top40 = anova_df.head(40).feature.tolist()
corr_matrix = sampled[top40].corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(corr_matrix, mask=mask, annot=False, cmap='RdBu_r',
            center=0, vmin=-1, vmax=1,
            linewidths=0.3, linecolor='#e0e0e0', ax=ax,
            cbar_kws={'shrink': 0.6})
ax.set_title('Feature-feature correlation — top 40 by ANOVA F-score', fontsize=12, fontweight='bold')
ax.tick_params(axis='both', labelsize=7)
plt.tight_layout()
plt.show()

# Highly correlated pairs (redundant features)
high_corr = []
for i in range(len(top40)):
    for j in range(i+1, len(top40)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.90:
            high_corr.append((top40[i], top40[j], round(r, 3)))

print(f'Highly correlated pairs (|r|>0.90): {len(high_corr)}')
for a, b, r in high_corr[:15]:
    print(f'  {a}  ↔  {b}  :  r={r}')

## 11 · PCA — variance analysis & scree plot

In [ ]:
X_clean  = sampled[feature_cols].fillna(sampled[feature_cols].median())
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_clean)
y        = sampled[TARGET_COL].values

pca_full = PCA(random_state=RANDOM_STATE)
pca_full.fit(X_scaled)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)

n80 = np.searchsorted(cumvar, 0.80) + 1
n90 = np.searchsorted(cumvar, 0.90) + 1
n95 = np.searchsorted(cumvar, 0.95) + 1

print(f'Total features          : {len(feature_cols)}')
print(f'PC1+PC2 explained var   : {cumvar[1]*100:.1f}%')
print(f'Components for 80%      : {n80}')
print(f'Components for 90%      : {n90}')
print(f'Components for 95%      : {n95}')

In [ ]:
n_show = min(80, len(feature_cols))
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree
axes[0].bar(range(1, n_show+1),
            pca_full.explained_variance_ratio_[:n_show] * 100,
            color='#3B8BD4', alpha=0.8)
axes[0].set_xlabel('Principal component')
axes[0].set_ylabel('Explained variance (%)')
axes[0].set_title(f'Scree plot (first {n_show} PCs)')

# Cumulative
axes[1].plot(range(1, n_show+1), cumvar[:n_show]*100, color='#3B8BD4', lw=2)
for thr, n, col in [(0.80,n80,'#5DCAA5'),(0.90,n90,'#EF9F27'),(0.95,n95,'#E85D24')]:
    axes[1].axhline(thr*100, ls='--', color=col, alpha=0.7, label=f'{int(thr*100)}% → {n} PCs')
    axes[1].axvline(n,        ls=':',  color=col, alpha=0.5)
axes[1].set_xlabel('Number of components')
axes[1].set_ylabel('Cumulative explained variance (%)')
axes[1].set_title('Cumulative variance')
axes[1].legend(fontsize=9)

plt.suptitle('PCA variance analysis (skeleton-sampled ROIs, all features)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# PCA per feature group — where does the variance come from?
print(f'{"Group":<45} {"n_features":>10} {"PC1+PC2 %":>12}')
print('-' * 70)

groups = [
    ('raw / global',  meta_df[(meta_df.source=='raw') & (meta_df.region=='global')].index),
    ('raw / tiles',   meta_df[(meta_df.source=='raw') & (meta_df.region.isin(TILE_LABELS))].index),
    ('gabor / global',meta_df[(meta_df.source=='gabor') & (meta_df.region=='global')].index),
    ('gabor / tiles', meta_df[(meta_df.source=='gabor') & (meta_df.region.isin(TILE_LABELS))].index),
    ('width_ratio',   pd.Index(['width_ratio'])),
]

for label, idx in groups:
    cols_g = idx.tolist()
    if len(cols_g) < 2:
        print(f'{label:<45} {len(cols_g):>10}   (single feature)')
        continue
    Xg   = StandardScaler().fit_transform(
        sampled[cols_g].fillna(sampled[cols_g].median()))
    pca_g = PCA(n_components=2).fit(Xg)
    var2  = pca_g.explained_variance_ratio_.sum() * 100
    print(f'{label:<45} {len(cols_g):>10} {var2:>11.1f}%')

In [ ]:
# 2D PCA scatter coloured by class
pca2 = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca2 = pca2.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(9, 7))
for lbl, name, color, alpha, s in [
    (0, 'Healthy',  '#5DCAA5', 0.25, 12),
    (1, 'Stenosis', '#E85D24', 0.80, 25),
]:
    mask = y == lbl
    ax.scatter(X_pca2[mask, 0], X_pca2[mask, 1],
               c=color, label=name, alpha=alpha, s=s, linewidths=0)
ax.set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('PCA 2D scatter — skeleton-sampled ROIs')
ax.legend(markerscale=2)
plt.tight_layout()
plt.show()

## 12 · Gabor filter analysis — which orientations & scales are most discriminative?

In [ ]:
# Mean F-score per filter index (average over all 30 stats per filter)
filter_scores = np.zeros(N_FILTERS)
for i in range(N_FILTERS):
    filter_cols = [c for c in feature_cols if f'gabor_f{i}_' in c]
    if filter_cols:
        idxs = anova_df[anova_df.feature.isin(filter_cols)].F_score
        filter_scores[i] = idxs.mean()

# Reshape to orientations × sizes grid
score_grid = filter_scores.reshape(N_ORIENTATIONS, N_SIZES)

import numpy as np
thetas  = np.arange(0, 180, 180 / N_ORIENTATIONS)
lambdas = np.linspace(4, 20, N_SIZES)

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(score_grid, aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(N_SIZES))
ax.set_xticklabels([f'{l:.1f}' for l in lambdas])
ax.set_yticks(range(N_ORIENTATIONS))
ax.set_yticklabels([f'{t:.0f}°' for t in thetas])
ax.set_xlabel('Wavelength λ (pixels)')
ax.set_ylabel('Orientation θ')
ax.set_title('Mean ANOVA F-score per Gabor filter (orientation × scale)')
plt.colorbar(im, ax=ax, label='Mean F-score')
plt.tight_layout()
plt.show()

best_orient_idx = np.argmax(score_grid.mean(axis=1))
best_scale_idx  = np.argmax(score_grid.mean(axis=0))
print(f'Most discriminative orientation: {thetas[best_orient_idx]:.0f}°')
print(f'Most discriminative scale (λ)  : {lambdas[best_scale_idx]:.1f} px')

## 13 · Outlier detection

In [ ]:
z_scores     = np.abs(stats.zscore(X_clean, nan_policy='omit'))
outlier_mask = (z_scores > 4).any(axis=1)

print(f'Rows |z|>4 in any feature: {outlier_mask.sum()} ({outlier_mask.sum()/len(sampled)*100:.1f}%)')
print('By class:')
print(sampled[outlier_mask][TARGET_COL].value_counts().rename({0:'Healthy',1:'Stenosis'}))

# Top features producing outliers
out_per_feat = pd.Series((z_scores > 4).sum(axis=0), index=feature_cols)
print('\nTop 10 features with most outlier rows:')
print(out_per_feat.sort_values(ascending=False).head(10).to_string())

## 14 · Summary & next steps

In [ ]:
vc_s = sampled[TARGET_COL].value_counts()

print('=' * 65)
print('DATA EXPLORATION SUMMARY')
print('=' * 65)
print(f'Total ROIs in CSV              : {len(df):,}')
print(f'  Skeleton-sampled (roi_idx≤100): {len(sampled):,}')
print(f'  Forced GT (roi_idx>100)       : {len(forced):,}  ← exclude from test set')
print()
print(f'Skeleton-sampled class balance:')
print(f'  Healthy   : {vc_s.get(0,0):,}')
print(f'  Stenosis  : {vc_s.get(1,0):,}')
print(f'  Ratio     : {vc_s.get(0,0)/max(vc_s.get(1,1),1):.1f}:1')
print()
print(f'Total feature columns          : {len(feature_cols)}')
print(f'Missing values                 : {df[feature_cols].isna().sum().sum()}')
print(f'ANOVA sig. features (p<0.05)   : {sig} / {len(feature_cols)}')
print(f'PCA components for 80% var     : {n80}')
print(f'PCA components for 95% var     : {n95}')
print(f'Outlier rows (|z|>4)           : {outlier_mask.sum()}')
print(f'Patients with 0 stenosis ROIs  : {zero_stenosis_patients}')
print()
print('NEXT STEPS')
print('-' * 65)
print('1. Resolve with Bria: include/exclude patients with 0 stenosis ROIs?')
print('2. Create train/test split at PATIENT level (no leakage)')
print('3. For TEST set: drop all forced GT rows (roi_idx > 100)')
print('4. For TRAIN set: keep forced GT rows (extra positive signal)')
print('5. Handle NaNs via median imputation (fit on train, apply to test)')
print('6. Random downsampling on train set only')
print('7. Feature selection: RF importance + SelectKBest (ANOVA F-test)')
print('8. Hard negative mining on full unbalanced train set')
print('9. Baseline: Logistic Regression → XGBoost / RF / SVM / LightGBM')
print('10. GridSearchCV optimising F1-score; soft-voting ensemble if gains')